# Starter: A Simple Research Agent

This notebook builds a small **agentic workflow**: Claude decides what to search for, calls a web search tool (Tavily), reads the results, and may search again before writing a short report with sources.

**Before you run it**, create a `.env` file in this folder containing:

```
ANTHROPIC_API_KEY=your_key_here
TAVILY_API_KEY=your_key_here
```

Both services have their own sign-up pages for keys. Never share or upload your keys.

## Step 1: Imports and setup

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from tavily import TavilyClient

load_dotenv()

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
tavily = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

MODEL = 'claude-sonnet-5-5'

## Step 2: Define the tool

A tool is just a normal Python function. The agent can call it, and we send back whatever it returns.

In [ ]:
def web_search(query, max_results=5):
    # Search the web and return a compact text summary of the results
    response = tavily.search(query=query, max_results=max_results)
    lines = []
    for r in response['results']:
        lines.append('- ' + r['title'] + ' (' + r['url'] + '): ' + r['content'][:400])
    return '\n'.join(lines)

# Quick test of the tool on its own
print(web_search('what is agentic AI'))

## Step 3: Describe the tool to Claude

Claude can't see your Python code. It only sees this description, so a clear name and explanation help it decide when to use the tool.

In [ ]:
tools = [
    {
        'name': 'web_search',
        'description': 'Search the web for current information on a topic. Returns titles, URLs and short snippets.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'The search query'}
            },
            'required': ['query']
        }
    }
]

SYSTEM = (
    'You are a careful research assistant. Use the web_search tool as many times as you need. '
    'Then write a short report (under 300 words) and list your sources as URLs at the end.'
)

## Step 4: The agent loop

This is the core idea. We keep calling Claude. If it asks to use a tool, we run the tool, hand back the result, and call Claude again. When it stops asking for tools, its answer is final. `max_steps` stops the agent from looping forever.

In [ ]:
def run_research_agent(topic, max_steps=6):
    messages = [
        {'role': 'user', 'content': 'Research this topic and write a short report with sources: ' + topic}
    ]

    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            system=SYSTEM,
            tools=tools,
            messages=messages,
        )
        messages.append({'role': 'assistant', 'content': response.content})

        # No tool requested: Claude has finished, so return the text
        if response.stop_reason != 'tool_use':
            return ''.join(b.text for b in response.content if b.type == 'text')

        # Otherwise run every tool Claude asked for and send the results back
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                print('Searching:', block.input['query'])
                result = web_search(block.input['query'])
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': result,
                })
        messages.append({'role': 'user', 'content': tool_results})

    return 'Stopped: reached the maximum number of steps.'

## Step 5: Run it

Change the topic to anything you like.

In [ ]:
report = run_research_agent('How are companies using agentic AI workflows today?')
print()
print(report)

## Try next

1. **Reflection:** after the report is written, send it back to Claude with the prompt 'Critique this report and improve it', then compare the two versions.
2. **A second tool:** add a tool that fetches a full web page, so the agent can read more than snippets.
3. **Evals:** write 3 test topics and check whether each report includes sources and stays under 300 words.
4. **Print the steps:** print every search query the agent makes to see how it plans its research.